# BERTopic and Theme Taxonomy Design

Goal: build an interpretable, unsupervised thematic-analysis pipeline and use it to design a first theme taxonomy.

Day 3 found clusters with UMAP + HDBSCAN but left them as numbered groups that someone had to read to understand. BERTopic uses the same steps and adds **keywords for each topic**, which makes topics readable at scale. Then comes the part no algorithm does: turning topics into **themes**, which are named, defined and useful to the people who act on the feedback.

Three ideas that are easy to mix up (the key learning outcome for today):

| | Topic modelling | Thematic analysis | Classification |
|---|---|---|---|
| Question | Which groups of similar comments exist? | What do the groups mean, and which matter? | Which known theme does this comment belong to? |
| Labels | None: discovered from the data | Themes defined by an analyst, from the data plus domain knowledge | Fixed and given in advance |
| Who decides | The algorithm | The analyst | A model trained or prompted on the taxonomy |
| Output | Topics: keywords and example comments | A taxonomy: theme names, definitions, boundaries | One or more labels per comment |
| In this project | Day 3 clusters, BERTopic today | Today's taxonomy | Day 2 baseline, Day 5 LLM |

A topic is a statistical pattern. A theme is an interpretation. Topic modelling *proposes*; thematic analysis *decides*; classification *applies* the decision to new comments.

Explored: number of topics (8, 12, 15, 20), random seeds.

Measured: topic coherence, interpretability (keywords and representative comments), topic size, topic overlap, stability, alignment with the existing categories.

Input: `data/processed/cleaned_feedback.csv`. Outputs are written to `results/`.

## 0. Setup

In [1]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import umap
import yaml
from bertopic import BERTopic
from gensim.corpora import Dictionary
from gensim.models.coherencemodel import CoherenceModel
from hdbscan import HDBSCAN
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics import adjusted_rand_score
from sklearn.metrics.pairwise import cosine_similarity

RESULTS = Path("../results")
RESULTS.mkdir(exist_ok=True)
pd.set_option("display.max_colwidth", 120)

# Harmless library warning: a seeded UMAP runs single-threaded.
warnings.filterwarnings("ignore", message="n_jobs value")

/home/riccha/Documents/practice-agentic-ai/student-feedback-intelligence/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Load the data and embed it

All 724 comments, as on Day 3. The index is used as `comment_id`, so every topic and theme can point back to real comments (Day 5's grounded summaries need this).

Embeddings use `all-MiniLM-L6-v2`, the model chosen on Day 3. They are computed once and passed to BERTopic, so every run below reuses them.

In [2]:
df = pd.read_csv("../data/processed/cleaned_feedback.csv")
df.index.name = "comment_id"
texts = df["text"].tolist()

embedding_model = SentenceTransformer("all-MiniLM-L6-v2")
E = embedding_model.encode(texts, normalize_embeddings=True)
E.shape

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4190.80it/s]


(724, 384)

## 2. How BERTopic works

BERTopic is a pipeline of four swappable steps:

1. **Embed** each comment (SentenceTransformer)
2. **Reduce** the embeddings (UMAP)
3. **Cluster** them (HDBSCAN); comments in no dense region become topic −1 (outliers)
4. **Describe** each cluster with **c-TF-IDF** (class-based TF-IDF)

Steps 1–3 are exactly Day 3's Approach B, so the settings are the ones chosen there: UMAP `n_neighbors=15`, 5 dimensions, cosine; HDBSCAN `min_cluster_size=10`. The topics should look much like Day 3's 17 clusters.

Step 4 is new. c-TF-IDF joins all comments of a topic into one long document, then runs TF-IDF across topics instead of across comments. A word scores high when it is frequent **in this topic** and rare **in the other topics**. Those are the topic's keywords.

Two choices differ from BERTopic's defaults:
- **`hdbscan.HDBSCAN` with `prediction_data=True`** instead of sklearn's version. The `hdbscan` library can assign a *new* comment to an existing topic (`topic_model.transform`), which the production API will need on Day 7. sklearn's HDBSCAN can't.
- **`CountVectorizer(stop_words="english", min_df=2)`** builds the vocabulary for keywords. Stop words such as *the* and *is* would otherwise top every topic. `min_df=2` drops words seen in only one comment (mostly typos). Only single words: sklearn removes stop words *before* forming two-word phrases, so "very good, good" turns into the fake phrase "good good".

In [3]:
def make_model(min_cluster_size=10, seed=42, nr_topics=None):
    return BERTopic(
        embedding_model=embedding_model,
        umap_model=umap.UMAP(n_neighbors=15, n_components=5, metric="cosine", random_state=seed),
        hdbscan_model=HDBSCAN(min_cluster_size=min_cluster_size, prediction_data=True),
        vectorizer_model=CountVectorizer(stop_words="english", min_df=2),
        nr_topics=nr_topics,
    )


topic_model = make_model()
topics = np.array(topic_model.fit_transform(texts, E)[0])

topic_model.get_topic_info()[["Topic", "Count", "Name"]]

,Topic,Count,Name
0,-1,65,-1_yes_ok_bad_students
1,0,87,0_teaching_delivery_interaction_punctuality
2,1,77,1_course_content_courses_depth
3,2,69,2_pattern_paper_exam_examination
4,3,67,3_library_books_facilities_available
5,4,63,4_practical_work_labs_practicals
6,5,50,5_best_good_better_great
7,6,36,6_activities_lot_held_participate
8,7,36,7_excellent_nice_perfect_accurate
9,8,36,8_poor_bad_improve_descent


**Result:** 16 topics plus 65 outliers (9%), the same shape as Day 3's 17 clusters. The keywords already show that topics 0–4 and 6 are about a subject (teaching, courses, exams, library, labs, activities), while most of the rest are tone words (good, excellent, poor, average).

## 3. What's in each topic?

For each topic: **frequency** (count and share of all comments), **top keywords** (c-TF-IDF) and **representative comments**.

BERTopic's own representative comments often repeat ("Excellent", "EXCELLENT", "Excellent"). Here they are the 5 *distinct* comments closest to the topic's centre in embedding space, i.e. the most typical ones.

Read every topic and ask: can I name it? Is it a **topic** (library, exams) or only a **tone** ("good", "poor")?

In [4]:
info = topic_model.get_topic_info().query("Topic != -1")
summary = pd.DataFrame({
    "topic": info["Topic"],
    "count": info["Count"],
    "pct": (100 * info["Count"] / len(df)).round(1),
    "keywords": info["Representation"].apply(lambda words: ", ".join(w for w in words[:8] if w)),
}).set_index("topic")

rep_rows = []
for topic in summary.index:
    in_topic = topics == topic
    members = df[in_topic].assign(similarity=E[in_topic] @ E[in_topic].mean(axis=0))
    members = members.sort_values("similarity", ascending=False)
    members = members[~members["text"].str.lower().str.strip(" .").duplicated()].head(5)
    for rank, (comment_id, row) in enumerate(members.iterrows(), start=1):
        rep_rows.append({"topic": topic, "rank": rank, "comment_id": comment_id, "text": row["text"]})
representative = pd.DataFrame(rep_rows)

for topic, group in representative.groupby("topic"):
    print(f"--- topic {topic} ({summary.loc[topic, 'count']} comments, {summary.loc[topic, 'pct']}%)")
    print(f"    keywords: {summary.loc[topic, 'keywords']}")
    for text in group["text"]:
        print("     ", text[:150])

--- topic 0 (87 comments, 12.0%)
    keywords: teaching, delivery, interaction, punctuality, teachers, lectures, teacher, punctual
      Interaction between students and teachers is good ,The lectures delivery is fine, punctuality is good
      Teaching is excellent,but lecture delivery is not up to the mark.Punctuality is good
      in terms of lecture delivery that is good.in terms of punctuality that is also good but in terms of intraction that is very bad. There are some teache
      Excellent lectures are delivered by teachers and all teachers are very punctual.
      Interaction is satisfying, Lecture delivery is Good, Punctuality is Excellent
--- topic 1 (77 comments, 10.6%)
    keywords: course, content, courses, depth, material, knowledge, proper, subjects
      course contents are very much relevant to our courses
      content of the course is good
      Content of the courses is good but some courses need improvement. Course material is not upto the mark.
      The content 

**Result:** six topics (0–4, 6) are clear subjects you can name from the keywords alone. Topics 5 and 7–15 are the same message ("good", "excellent", "poor", "average") in different words. Topics 13 (praise for the university) and 15 ("they are doing their job well") look like topics but have no subject.

### 3a. Topic descriptions

The plan asks for a description of every topic. This is the human step: read the keywords and comments above, then write a short description and decide which **theme** the topic belongs to (section 7).

These are draft descriptions written from the output above. **Check and rewrite them**: you are the analyst. Topic numbers come from a seeded run, but if a rerun renumbers them, update this table.

Topics that are only a tone ("good", "excellent", "poor") go to `GENERAL`: they carry sentiment but no theme.

In [5]:
GENERAL = "General feedback (no specific theme)"

TOPICS = {
    0: ("Lecture delivery, interaction with students and punctuality", "Teaching Quality"),
    1: ("Depth of courses and quality of course material", "Course Content"),
    2: ("Exam pattern, marks distribution and paper checking", "Assessment and Marking"),
    3: ("Library facilities and availability of books", "Library and Learning Resources"),
    4: ("Labs and practical work", "Practical Work"),
    5: ("Generic praise: good, best, great", GENERAL),
    6: ("Extracurricular activities and events", "Extracurricular Activities"),
    7: ("One-word praise: excellent, nice, perfect", GENERAL),
    8: ("Generic criticism: poor, bad, needs improvement", GENERAL),
    9: ("'Good'", GENERAL),
    10: ("'Very good' / 'pretty good'", GENERAL),
    11: ("'Everything is fine'", GENERAL),
    12: ("'Satisfactory' / 'super'", GENERAL),
    13: ("Praise for the university as a whole", GENERAL),
    14: ("'Average'", GENERAL),
    15: ("'They are doing their job well'", GENERAL),
}

summary["description"] = summary.index.map(lambda t: TOPICS[t][0])
summary["theme"] = summary.index.map(lambda t: TOPICS[t][1])
summary[["count", "pct", "keywords", "description", "theme"]]

,count,pct,keywords,description,theme
topic,,,,,
0,87,12.0,"teaching, delivery, interaction, punctuality, teachers, lectures, teacher, punctual","Lecture delivery, interaction with students and punctuality",Teaching Quality
1,77,10.6,"course, content, courses, depth, material, knowledge, proper, subjects",Depth of courses and quality of course material,Course Content
2,69,9.5,"pattern, paper, exam, examination, marks, distribution, exams, good","Exam pattern, marks distribution and paper checking",Assessment and Marking
3,67,9.3,"library, books, facilities, available, number, increased, problem, think",Library facilities and availability of books,Library and Learning Resources
4,63,8.7,"practical, work, labs, practicals, hours, knowledge, know, working",Labs and practical work,Practical Work
5,50,6.9,"best, good, better, great, quite, gud, fine, really","Generic praise: good, best, great",General feedback (no specific theme)
6,36,5.0,"activities, lot, held, participate, university, events, students, platform",Extracurricular activities and events,Extracurricular Activities
7,36,5.0,"excellent, nice, perfect, accurate, great, awesome, superb, simply","One-word praise: excellent, nice, perfect",General feedback (no specific theme)
8,36,5.0,"poor, bad, improve, descent, good, dont, strict, improved","Generic criticism: poor, bad, needs improvement",General feedback (no specific theme)


**Result:** 6 of the 16 topics are real themes and hold 399 comments (55%). The other 10 topics are general feedback (260 comments, 36%), and 65 comments (9%) are outliers.

## 4. Evaluating a topic model

Each topic model below is scored the same way:
- **n_topics**, **largest_topic_pct**, **outlier_pct**: as on Day 3. One huge topic is a catch-all; outliers (topic −1) have no topic
- **coherence_c_v**: do a topic's top-10 keywords belong together? For every keyword it records how strongly it co-occurs with each of the other keywords, beyond chance (NPMI). Then it checks how similar the keywords' co-occurrence patterns are (cosine). It ranges from 0 to 1, and higher is better. Among the standard coherence measures it matches human judgements of topics best. Compare models on the same data rather than reading the number on its own
- **diversity**: share of *unique* words among all topics' top-10 keywords. 1 = no keyword appears in two topics, low = topics overlap
- **ari_vs_category**: as on Day 3, alignment with the dataset's 6 categories

Coherence measures whether keywords are used *the same way*, not whether the topic is useful to anyone. Keep that in mind when topics made of one-word comments ("Excellent", "Perfect") get scored.

In [6]:
analyzer = CountVectorizer(stop_words="english", min_df=2).build_analyzer()
tokens = [analyzer(text) for text in texts]
dictionary = Dictionary(tokens)


def topic_words(model, topics):
    # The keyword list of every topic, skipping the outliers (-1).
    all_words = []
    for topic in sorted(set(topics)):
        if topic == -1:
            continue
        # get_topic gives (word, score) pairs. Small topics are padded with empty words "", which we drop.
        words = [word for word, score in model.get_topic(topic) if word != ""]
        all_words.append(words)
    return all_words


def evaluate(model, topics):
    words = topic_words(model, topics)
    clustered = topics != -1
    return {
        "n_topics": len(words),
        "largest_topic_pct": round(100 * pd.Series(topics[clustered]).value_counts(normalize=True).max(), 1),
        "outlier_pct": round(100 * (~clustered).mean(), 1),
        "coherence_c_v": CoherenceModel(topics=words, texts=tokens, dictionary=dictionary, coherence="c_v").get_coherence(),
        "diversity": len({w for ws in words for w in ws}) / sum(len(ws) for ws in words),
        "ari_vs_category": adjusted_rand_score(df["category"], topics),
    }


pd.DataFrame([{"model": "BERTopic, min_cluster_size=10", **evaluate(topic_model, topics)}]).round(3)

,model,n_topics,largest_topic_pct,outlier_pct,coherence_c_v,diversity,ari_vs_category
0,"BERTopic, min_cluster_size=10",16,13.2,9.0,0.496,0.806,0.336


**Result:** coherence is 0.496 and diversity 0.806, so about a fifth of the keyword slots repeat a word another topic also uses ("good", "best", "held"). ARI vs category is 0.336, close to Day 3's chosen UMAP + HDBSCAN run (0.326), as expected from the same pipeline with a different HDBSCAN implementation.

### 4a. Coherence per topic

Which topics have keywords that belong together? Compare coherence with the descriptions.

In [7]:
summary["coherence_c_v"] = CoherenceModel(
    topics=topic_words(topic_model, topics), texts=tokens, dictionary=dictionary, coherence="c_v"
).get_coherence_per_topic()

summary[["count", "keywords", "description", "coherence_c_v"]].sort_values("coherence_c_v", ascending=False).round(3)

,count,keywords,description,coherence_c_v
topic,,,,
2,69,"pattern, paper, exam, examination, marks, distribution, exams, good","Exam pattern, marks distribution and paper checking",0.670
0,87,"teaching, delivery, interaction, punctuality, teachers, lectures, teacher, punctual","Lecture delivery, interaction with students and punctuality",0.600
12,19,"satisfactory, super, satisfied, appreciable, excllent, extraordinary, felt, held",'Satisfactory' / 'super',0.585
5,50,"best, good, better, great, quite, gud, fine, really","Generic praise: good, best, great",0.568
7,36,"excellent, nice, perfect, accurate, great, awesome, superb, simply","One-word praise: excellent, nice, perfect",0.568
1,77,"course, content, courses, depth, material, knowledge, proper, subjects",Depth of courses and quality of course material,0.551
15,13,"extraordinary, job, able, help, conducted, doing, hard, going",'They are doing their job well',0.538
14,15,"average, rating, term, based, subjects",'Average',0.535
6,36,"activities, lot, held, participate, university, events, students, platform",Extracurricular activities and events,0.526


**Result:** coherence doesn't track usefulness here. The exam and teaching topics score highest (0.67, 0.60), but the one-word praise topics 12, 5 and 7 (0.57–0.59) outscore the library (0.43) and labs (0.39) topics, whose keywords include filler words such as "think", "problem" and "know". Coherence alone can't tell a theme from a tone.

### 4b. Topic overlap

Which topics are closest in meaning? The similarity of two topics is the cosine similarity of their centres (mean embedding of their comments). The most similar pairs are candidates for merging.

In [8]:
topic_ids = list(summary.index)
centres = np.array([E[topics == t].mean(axis=0) for t in topic_ids])
similarity = pd.DataFrame(cosine_similarity(centres), index=topic_ids, columns=topic_ids)

pairs = similarity.where(np.triu(np.ones(similarity.shape, dtype=bool), k=1)).stack().nlargest(8)
pd.DataFrame({
    "topic_a": pairs.index.get_level_values(0),
    "topic_b": pairs.index.get_level_values(1),
    "similarity": pairs.values.round(3),
    "description_a": summary.loc[pairs.index.get_level_values(0), "description"].values,
    "description_b": summary.loc[pairs.index.get_level_values(1), "description"].values,
})

,topic_a,topic_b,similarity,description_a,description_b
0,7,9,0.767,"One-word praise: excellent, nice, perfect",'Good'
1,7,10,0.724,"One-word praise: excellent, nice, perfect",'Very good' / 'pretty good'
2,9,10,0.719,'Good','Very good' / 'pretty good'
3,5,10,0.711,"Generic praise: good, best, great",'Very good' / 'pretty good'
4,5,8,0.692,"Generic praise: good, best, great","Generic criticism: poor, bad, needs improvement"
5,8,10,0.687,"Generic criticism: poor, bad, needs improvement",'Very good' / 'pretty good'
6,0,1,0.676,"Lecture delivery, interaction with students and punctuality",Depth of courses and quality of course material
7,6,13,0.663,Extracurricular activities and events,Praise for the university as a whole


**Result:** the 6 most similar pairs are all general-feedback topics (similarity 0.69–0.77), confirming they're one bucket split by wording. Even generic praise and generic criticism (5 and 8) are close: the embedding groups them by being short and evaluative, not by sentiment. The closest pair of real themes is teaching and course content (0.68). They're related, but different people act on them, so they should stay separate.

### 4c. Topics vs existing categories

Does the discovered structure line up with the 6 categories? A topic that recovers a category has most of its comments in one column.

In [9]:
pd.crosstab(topics, df["category"], rownames=["topic"])

category,course_content,examination,extracurricular,labwork,library_facilities,teaching
topic,,,,,,
-1,10,9,19,9,6,12
0,6,1,4,2,0,74
1,72,1,1,0,3,0
2,1,62,1,4,1,0
3,0,0,1,1,65,0
4,0,1,0,62,0,0
5,7,5,13,7,10,8
6,0,0,35,0,0,1
7,7,5,9,4,5,6


**Result:** each real-theme topic maps almost entirely onto one category (0 → teaching 74/87, 1 → course content 72/77, 2 → examination 62/69, 3 → library 65/67, 4 → labwork 62/63, 6 → extracurricular 35/36). Every general-feedback topic is spread across all six categories, which is why ARI stays at 0.34.

## 5. Theme granularity: 8, 12, 15 or 20 topics?

`nr_topics=n` makes BERTopic merge topics until n remain (outliers count as one of the n). It repeatedly joins the two topics with the most similar c-TF-IDF keywords.

Start from a finer model (`min_cluster_size=5`, about 23 topics) so there is something to merge, then reduce it to 8, 12, 15 and 20. The topic names show *what* got merged.

In [10]:
granularity_rows = []
for n in [8, 12, 15, 20]:
    model = make_model(min_cluster_size=5, nr_topics=n)
    reduced_topics = np.array(model.fit_transform(texts, E)[0])
    granularity_rows.append({"nr_topics": n, **evaluate(model, reduced_topics)})
    print(f"--- nr_topics={n}")
    print("   ", model.get_topic_info()["Name"].tolist())

pd.DataFrame(granularity_rows).round(3)

--- nr_topics=8
    ['-1_students_like_accurate_understand', '0_university_good_books_course', '1_good_excellent_bad_ok', '2_pattern_paper_exam_marks', '3_good_thing_fine_things', '4_satisfactory_super_satisfied_appreciable', '5_average_rating_term_based', '6_yes___']
--- nr_topics=12
    ['-1_students_accurate_like_understand', '0_university_course_activities_good', '1_good_excellent_bad_ok', '2_practical_work_labs_practicals', '3_pattern_paper_exam_marks', '4_library_books_facilities_available', '5_good_thing_fine_things', '6_average_rating_term_based', '7_satisfactory_satisfied_appreciable_felt', '8_super_excllent_extraordinary_held', '9_yes___', '10_best_better__']
--- nr_topics=15
    ['-1_students_accurate_like_understand', '0_course_teaching_delivery_interaction', '1_bad_good_poor_quite', '2_excellent_ok_nice_perfect', '3_practical_work_labs_practicals', '4_pattern_paper_exam_examination', '5_library_books_facilities_available', '6_activities_university_held_lot', '7_fine_things

,nr_topics,n_topics,largest_topic_pct,outlier_pct,coherence_c_v,diversity,ari_vs_category
0,8,7,42.2,6.1,0.543,0.893,0.130
1,12,11,32.4,6.1,0.465,0.910,0.221
2,15,14,23.8,6.1,0.463,0.806,0.286
3,20,19,12.8,6.1,0.466,0.773,0.326


**Result:** forcing fewer topics merges the *real* themes first. At 8, library and course comments end up in one topic holding 42% of comments, and teaching, labs and activities disappear from the topic names, while tiny tone topics ("yes", "average", "satisfactory") survive. ARI falls to 0.13. Merging follows keyword similarity, and a topic whose only keyword is "yes" is similar to nothing, so it never gets merged. Only 20 keeps the themes apart. Coherence is highest at 8, so here it rewards the worst model.

## 6. Stability across random seeds

Same settings as section 2, 5 UMAP seeds. Compare each run with seed 0 (ARI) as on Day 3.

In [11]:
seed_topics = [np.array(make_model(seed=seed).fit_transform(texts, E)[0]) for seed in range(5)]

stability = pd.DataFrame({
    "seed": range(5),
    "n_topics": [len(set(t) - {-1}) for t in seed_topics],
    "outlier_pct": [round(100 * (t == -1).mean(), 1) for t in seed_topics],
    "ari_vs_seed_0": [round(adjusted_rand_score(seed_topics[0], t), 3) for t in seed_topics],
})
stability

,seed,n_topics,outlier_pct,ari_vs_seed_0
0,0,16,7.0,1.000
1,1,16,9.9,0.882
2,2,16,9.9,0.855
3,3,16,10.8,0.857
4,4,16,9.8,0.876


**Result:** the number of topics never changes (16), but ARI against seed 0 is 0.86–0.88, a little lower than Day 3's 0.91. Topic boundaries, and therefore the topic numbers in `TOPICS`, shift between seeds, so the taxonomy has to be tied to theme definitions, not to topic ids.

## 7. Theme taxonomy v1

A taxonomy is the analyst's decision about which themes exist, what each one means and where its boundaries are. It is built *from* the topics but is not the same as them:
- **Several topics can map to one theme.** All the tone-only topics become one `GENERAL` bucket.
- **One topic can hold several sub-themes.** The exam topic mixes exam *design* (pattern, marks distribution) with *marking* (paper checking errors), which are different problems owned by different people.
- **Themes need definitions** so that a human or an LLM (Day 5) can apply them consistently to new comments.
- **Themes the data doesn't support are left out**, however plausible they sound.

### 7a. The plan's candidate themes vs this data

The plan suggests themes such as Assessment Workload, Course Organisation, Technology and Student Support. No topic matches them. A keyword search checks whether they're rare or completely absent.

In [12]:
candidate_keywords = {
    "Assessment Workload": "workload|deadline|assignment|submission",
    "Course Organisation": "timetable|schedule|organi[sz]|timing",
    "Technology": "wifi|wi-fi|internet|computer|online|software|technolog",
    "Student Support": "support|counsel|mentor|guidance|help desk",
    "Feedback and Marking": "checking|recheck|answer key|evaluation",
}
pd.Series({theme: df["text"].str.contains(pattern, case=False).sum() for theme, pattern in candidate_keywords.items()}, name="comments")

Assessment Workload      0
Course Organisation      6
Technology               6
Student Support          1
Feedback and Marking    41
Name: comments, dtype: int64

**Result:** no comment mentions workload, deadlines or assignments. Technology (6), organisation (6) and student support (1) are too rare to form a theme, so they stay out of v1 as `not_observed`. Paper checking and evaluation (41) are common enough to be a sub-theme of Assessment and Marking.

### 7b. Themes, definitions and sub-themes

Draft definitions, based on the topics above. **Review them**: add, merge, rename or reject themes. Each theme lists `source_topics` (filled in from `TOPICS`) so the taxonomy stays traceable to the evidence.

In [13]:
TAXONOMY = {
    "Teaching Quality": {
        "description": "How teachers deliver lectures, interact with students, keep to time and know their subject.",
        "subthemes": ["Lecture delivery", "Interaction with students", "Punctuality", "Subject knowledge"],
    },
    "Course Content": {
        "description": "Depth, coverage and quality of courses and course material.",
        "subthemes": ["Depth of content", "Course material"],
    },
    "Assessment and Marking": {
        "description": "Exam pattern and question types, marks distribution, and accuracy and transparency of paper checking.",
        "subthemes": ["Assessment design", "Feedback and marking"],
    },
    "Library and Learning Resources": {
        "description": "Availability of books, issue limits and library facilities.",
        "subthemes": ["Book availability", "Issue limits and duration", "Library facilities"],
    },
    "Practical Work": {
        "description": "Labs and practical sessions: usefulness, duration and equipment.",
        "subthemes": ["Value of practicals", "Session length", "Equipment"],
    },
    "Extracurricular Activities": {
        "description": "Events, sports and activities outside the curriculum, and time to take part in them.",
        "subthemes": ["Range of activities", "Time to participate"],
    },
    GENERAL: {
        "description": "Short evaluative comments ('good', 'excellent', 'poor', 'average') with no topic. "
        "They carry sentiment, but their theme comes from the survey question, not from the text.",
        "subthemes": [],
    },
}

df["topic"] = topics
df["theme"] = df["topic"].map({t: theme for t, (_, theme) in TOPICS.items()}).fillna("Unassigned (outlier)")
pd.crosstab(df["theme"], df["category"], margins=True)

category,course_content,examination,extracurricular,labwork,library_facilities,teaching,All
theme,,,,,,,
Assessment and Marking,1,62,1,4,1,0,69
Course Content,72,1,1,0,3,0,77
Extracurricular Activities,0,0,35,0,0,1,36
General feedback (no specific theme),35,48,57,39,43,38,260
Library and Learning Resources,0,0,1,1,65,0,67
Practical Work,0,1,0,62,0,0,63
Teaching Quality,6,1,4,2,0,74,87
Unassigned (outlier),10,9,19,9,6,12,65
All,124,122,118,117,118,125,724


**Result:** the six actionable themes match the six existing categories almost one-to-one, so on this dataset the categories were already a sound top-level taxonomy. The analysis adds sub-themes, a general-feedback bucket (260 comments, 36%) and an outlier queue (65). General feedback is spread across every category, which suggests (unverified) that `category` is the survey question the student was answering. If so, a production system should keep the question with each comment: for "Excellent", it's the only clue to the theme.

### 7c. Write the taxonomy

Every theme has `status: proposed`. Nothing in it is final until an analyst accepts it: the same accept / merge / rename / reject flow the production system will use (Day 7).

In [14]:
theme_counts = df["theme"].value_counts()
taxonomy = {
    "version": "v1",
    "status": "draft, needs analyst review",
    "source": "notebooks/04_bertopic.ipynb: BERTopic, all-MiniLM-L6-v2, UMAP(15, 5, cosine, seed 42), HDBSCAN(min_cluster_size=10)",
    "themes": [
        {
            "name": name,
            "status": "proposed",
            "actionable": name != GENERAL,
            "description": spec["description"],
            "subthemes": spec["subthemes"],
            "source_topics": [int(t) for t, (_, theme) in TOPICS.items() if theme == name],
            "n_comments": int(theme_counts.get(name, 0)),
        }
        for name, spec in TAXONOMY.items()
    ],
    "outliers": {
        "n_comments": int(theme_counts.get("Unassigned (outlier)", 0)),
        "handling": "analyst review queue",
    },
    "not_observed": list(candidate_keywords)[:4],
}

(RESULTS / "theme_taxonomy_v1.yaml").write_text(yaml.safe_dump(taxonomy, sort_keys=False, allow_unicode=True))
print((RESULTS / "theme_taxonomy_v1.yaml").read_text())

version: v1
status: draft, needs analyst review
source: 'notebooks/04_bertopic.ipynb: BERTopic, all-MiniLM-L6-v2, UMAP(15, 5, cosine,
  seed 42), HDBSCAN(min_cluster_size=10)'
themes:
- name: Teaching Quality
  status: proposed
  actionable: true
  description: How teachers deliver lectures, interact with students, keep to time
    and know their subject.
  subthemes:
  - Lecture delivery
  - Interaction with students
  - Punctuality
  - Subject knowledge
  source_topics:
  - 0
  n_comments: 87
- name: Course Content
  status: proposed
  actionable: true
  description: Depth, coverage and quality of courses and course material.
  subthemes:
  - Depth of content
  - Course material
  source_topics:
  - 1
  n_comments: 77
- name: Assessment and Marking
  status: proposed
  actionable: true
  description: Exam pattern and question types, marks distribution, and accuracy and
    transparency of paper checking.
  subthemes:
  - Assessment design
  - Feedback and marking
  source_topics:
  -

## 9. Save outputs

In [16]:
summary.reset_index().round(3).to_csv(RESULTS / "topic_summary.csv", index=False)
representative.assign(description=representative["topic"].map(summary["description"])).to_csv(
    RESULTS / "representative_comments.csv", index=False
)
df[["text", "category", "sentiment", "topic", "theme"]].to_csv(RESULTS / "topic_assignments.csv")

report = f'''# Day 4: BERTopic Theme Discovery and Taxonomy

## Chosen model

{pd.DataFrame([evaluate(topic_model, topics)]).round(3).to_string(index=False)}

## Topics

{summary[["count", "pct", "coherence_c_v", "description", "theme"]].round(3).to_string()}

## Granularity

{pd.DataFrame(granularity_rows).round(3).to_string(index=False)}

## Stability

{stability.to_string(index=False)}

## Themes vs existing categories

{pd.crosstab(df["theme"], df["category"]).to_string()}

## Answers
{ANSWERS}'''

(RESULTS / "theme_taxonomy_analysis.md").write_text(report)

4720